<a href="https://colab.research.google.com/github/14-169-RaisyaSyifaSaleh/Kelompok11_Studio2/blob/main/Studio2_123140169_123140212.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kompetisi Klasifikasi Sampah**

---

### **1. Anggota Kelompok**
* **Anggota 1:** [Raisya Syifa Saleh] - [123140169]
* **Anggota 2:** [Diva Amelia Saputri] - [123140212]

---

### **2. Deskripsi Tugas**
Bangun model klasifikasi citra yang menerima satu foto sampah dan memprediksi salah satu dari lima kelas di bawah. Anda bebas memilih pendekatan: CNN from scratch seperti di Pertemuan 4, arsitektur modern dari Pertemuan 5, atau fine-tuning backbone pretrained ImageNet sesuai materi Studio 2. Data tambahan dari luar juga diperbolehkan.

Kelompok tidak pernah melihat validation set. Asisten dosen yang menjalankan model (model.pth) kelompok Anda pada validation set menggunakan skrip evaluate.py yang sama persis dengan yang ada di starter kit, lalu mencatat skornya di Google Sheets skor kelompok.

---



In [ ]:
import os
import sys
import torch
import torchvision
from PIL import Image

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))
    print("Memori GPU:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("GPU belum aktif. Periksa Runtime > Change runtime type.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU tersedia: True
Nama GPU: Tesla T4
Memori GPU: 14.56 GB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

dataset_folder = "/content/drive/MyDrive/Studio2_Kelompok11"

print("Lokasi dataset:", dataset_folder)
print("Folder ditemukan:", os.path.exists(dataset_folder))

if os.path.exists(dataset_folder):
    print("\nIsi folder:")
    for item in os.listdir(dataset_folder):
        print("-", item)
else:
    print("Folder belum ditemukan. Periksa lokasi Google Drive.")

Lokasi dataset: /content/drive/MyDrive/Studio2_Kelompok11
Folder ditemukan: True

Isi folder:
- dataset_mahasiswa


In [ ]:
import os

dataset_folder = "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"

print("Lokasi dataset:", dataset_folder)
print("Folder ditemukan:", os.path.exists(dataset_folder))

if os.path.exists(dataset_folder):
    print("\nIsi folder dataset_mahasiswa:")
    for item in os.listdir(dataset_folder):
        print("-", item)
else:
    print("Folder tidak ditemukan.")

Lokasi dataset: /content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa
Folder ditemukan: True

Isi folder dataset_mahasiswa:
- train_sources.csv
- BACA_SAYA.txt
- train
- starter_kit
- local_split


In [ ]:
from pathlib import Path

dataset_folder = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)

train_dir = dataset_folder / "train"

expected_classes = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam"
]

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Folder train ditemukan:", train_dir.exists())
print("\nIsi folder train:")

if train_dir.exists():
    for item in sorted(train_dir.iterdir()):
        print("-", item.name)

    print("\nJumlah gambar per kelas:")

    for class_name in expected_classes:
        class_dir = train_dir / class_name

        if class_dir.exists():
            images = [
                file for file in class_dir.rglob("*")
                if file.is_file()
                and file.suffix.lower() in image_extensions
            ]

            print(f"{class_name}: {len(images)} gambar")
        else:
            print(f"{class_name}: FOLDER TIDAK DITEMUKAN")
else:
    print("Folder train tidak ditemukan. Periksa kembali lokasi dataset.")

Folder train ditemukan: True

Isi folder train:
- botol_plastik
- gelas_plastik
- kertas_kardus
- saset_kemasan
- styrofoam

Jumlah gambar per kelas:
botol_plastik: 523 gambar
gelas_plastik: 537 gambar
kertas_kardus: 365 gambar
saset_kemasan: 673 gambar
styrofoam: 671 gambar


In [ ]:
from pathlib import Path

starter_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/"
    "dataset_mahasiswa/starter_kit"
)

print("Starter kit ditemukan:", starter_dir.exists())

if starter_dir.exists():
    print("\nDaftar file starter kit:")

    for item in sorted(starter_dir.rglob("*")):
        if item.is_file():
            print("-", item.relative_to(starter_dir))

Starter kit ditemukan: True

Daftar file starter kit:
- README_TEMPLATE.md
- evaluate.py
- submission/model.py
- train_baseline.py


In [ ]:
from pathlib import Path

dataset_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)

files_to_read = [
    dataset_dir / "BACA_SAYA.txt",
    dataset_dir / "starter_kit" / "submission" / "model.py",
    dataset_dir / "starter_kit" / "README_TEMPLATE.md",
]

for file_path in files_to_read:
    print("\n" + "=" * 60)
    print("FILE:", file_path.name)
    print("=" * 60)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File tidak ditemukan:", file_path)



FILE: BACA_SAYA.txt
STUDIO 2 — KOMPETISI KLASIFIKASI SAMPAH
Deep Learning (IF25-40401) · Teknik Informatika ITERA · Ganjil 2026/2027

Halaman kompetisi (aturan, jadwal, penilaian):
  https://informatika-itera.github.io/deep-learning-IF25-40401/studio-2/

Isi:
  train/<kelas>/*.jpg   train set, 5 kelas:
                        0 botol_plastik, 1 gelas_plastik, 2 kertas_kardus,
                        3 saset_kemasan, 4 styrofoam
  train_sources.csv     sumber (URL) setiap gambar
  starter_kit/          evaluate.py (skrip evaluasi resmi), train_baseline.py,
                        submission/model.py (contoh), README_TEMPLATE.md

Mulai cepat:
  cd starter_kit
  python train_baseline.py --data ../train --epochs 8
  python evaluate.py --submission ./submission --data ./my_val

Validation set tidak dibagikan. Asisten dosen menjalankan model.pth kelompok
Anda dengan evaluate.py yang sama pada setiap tenggat preview.

Gambar dikumpulkan dari internet untuk keperluan pendidikan non-komersial.

In [ ]:
from pathlib import Path

starter_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/"
    "dataset_mahasiswa/starter_kit"
)

for filename in ["train_baseline.py", "evaluate.py"]:
    file_path = starter_dir / filename

    print("\n" + "=" * 60)
    print(f"FILE: {filename}")
    print("=" * 60)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File tidak ditemukan:", file_path)


FILE: train_baseline.py
"""Baseline sederhana Studio 2: fine-tuning ResNet-18 (pretrained ImageNet).

Titik awal saja — kelompok diharapkan melakukan eksperimen sendiri
(backbone lain, augmentasi, freezing, scheduler, data tambahan, dll.).

    python train_baseline.py --data ./train --epochs 8
    python train_baseline.py --data ./train --epochs 8 --scratch   # tanpa bobot pretrained

Hasil: submission/model.pth + folder my_val/ (split validasi lokal 15%)
       yang bisa langsung diuji dengan:
    python evaluate.py --submission ./submission --data ./my_val
"""
import argparse
import random
import shutil
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from submission.model import get_model, get_transform


def make_local_split(src, dst, frac, seed):
    """Salin 15% gambar per kelas dari train ke my_val/ (sekali saja), sisanya ke my_train/."""
    dst_tr, dst_va = Path(dst) / "

In [ ]:
from pathlib import Path
from collections import Counter

dataset_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)
train_dir = dataset_dir / "train"

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Format gambar per kelas:\n")

for class_dir in sorted(train_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    extensions = Counter(
        p.suffix.lower()
        for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    )

    print(f"{class_dir.name}:")
    for ext, count in sorted(extensions.items()):
        print(f"  {ext}: {count}")

Format gambar per kelas:

botol_plastik:
  .jpg: 523
gelas_plastik:
  .jpg: 537
kertas_kardus:
  .jpg: 365
saset_kemasan:
  .jpg: 673
styrofoam:
  .jpg: 671


In [ ]:

from pathlib import Path
from collections import Counter
import random
import shutil

# Lokasi dataset di Google Drive
DATASET_DIR = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)
SOURCE_DIR = DATASET_DIR / "train"

# Lokasi hasil pembagian data
SPLIT_DIR = DATASET_DIR / "local_split"
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"

# Pengaturan pembagian data
SEED = 42
VAL_RATIO = 0.15

CLASSES = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Pastikan dataset sumber tersedia
assert SOURCE_DIR.exists(), (
    f"Folder dataset tidak ditemukan: {SOURCE_DIR}"
)

# Jangan menimpa pembagian sebelumnya secara tidak sengaja
if SPLIT_DIR.exists():
    # Periksa apakah hasil pembagian sebelumnya sudah lengkap
    if (
        TRAIN_DIR.is_dir()
        and VAL_DIR.is_dir()
        and all((TRAIN_DIR / c).is_dir() for c in CLASSES)
        and all((VAL_DIR / c).is_dir() for c in CLASSES)
    ):
        print("Folder hasil pembagian sudah ada.")
        print("Menggunakan hasil pembagian sebelumnya.")

    else:
        raise FileExistsError(
            f"Folder {SPLIT_DIR} sudah ada, tetapi hasil "
            "pembagian belum lengkap. Periksa folder sebelumnya "
            "sebelum membuat ulang."
        )

else:
    random.seed(SEED)

    # Pastikan semua kelas ditemukan
    for class_name in CLASSES:
        class_dir = SOURCE_DIR / class_name
        assert class_dir.is_dir(), (
            f"Folder kelas tidak ditemukan: {class_dir}"
        )

    # Buat folder hasil pembagian
    for class_name in CLASSES:
        (TRAIN_DIR / class_name).mkdir(parents=True)
        (VAL_DIR / class_name).mkdir(parents=True)

    # Bagi gambar di setiap kelas
    for class_name in CLASSES:
        class_dir = SOURCE_DIR / class_name

        images = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        )

        random.shuffle(images)

        val_count = round(len(images) * VAL_RATIO)
        val_images = images[:val_count]
        train_images = images[val_count:]

        for image_path in train_images:
            shutil.copy2(
                image_path,
                TRAIN_DIR / class_name / image_path.name
            )

        for image_path in val_images:
            shutil.copy2(
                image_path,
                VAL_DIR / class_name / image_path.name
            )

# Tampilkan hasil pembagian
print("HASIL PEMBAGIAN DATASET")
print("-" * 45)

total_train = 0
total_val = 0

for class_name in CLASSES:
    n_train = len(list((TRAIN_DIR / class_name).glob("*")))
    n_val = len(list((VAL_DIR / class_name).glob("*")))

    total_train += n_train
    total_val += n_val

    print(
        f"{class_name:18s} | "
        f"Train: {n_train:4d} | Val: {n_val:3d}"
    )

print("-" * 45)
print(f"Total training   : {total_train}")
print(f"Total validasi   : {total_val}")
print(f"Total keseluruhan: {total_train + total_val}")


Folder hasil pembagian sudah ada.
Menggunakan hasil pembagian sebelumnya.
HASIL PEMBAGIAN DATASET
---------------------------------------------
botol_plastik      | Train:  445 | Val:  78
gelas_plastik      | Train:  456 | Val:  81
kertas_kardus      | Train:  310 | Val:  55
saset_kemasan      | Train:  572 | Val: 101
styrofoam          | Train:  570 | Val: 101
---------------------------------------------
Total training   : 2353
Total validasi   : 416
Total keseluruhan: 2769
